# Classification Metrics

This notebook is intentionally self-contained. It does not import local utility files, so the statistical idea, simulation, Python functions, evaluation, plots, and exam translation are all visible in one place.

## What problem are we solving?

Classification metrics summarize how predicted class labels compare to true labels.

**Highest-probability exam pattern:** Given true and predicted labels, compute accuracy, confusion matrix entries, precision, recall, specificity, and interpret which error type matters.

## Assumptions, intuition, and theory

Accuracy can hide asymmetric mistakes. Confusion matrices separate false positives and false negatives. Precision answers “among predicted positives, how many were truly positive?” Recall answers “among true positives, how many did we catch?”

## Python method notes and documentation

`confusion_matrix` produces counts, `classification_report` summarizes per-class metrics, `precision_score` and `recall_score` compute named rates, and train/test splitting keeps evaluation honest.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)
- [confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
- [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
- [make_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html)
- [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html)
- [make_blobs](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html)

## Fully self-contained worked example

Every code line below is commented so it is easy to scan under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for metric summaries.
import matplotlib.pyplot as plt  # Import plotting tools.
from sklearn.base import clone  # Import clone for fresh estimator fits.
from sklearn.datasets import make_classification, make_moons  # Import classification simulators.
from sklearn.linear_model import LogisticRegression  # Import logistic regression classifier.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score  # Import explicit classification metrics.
from sklearn.model_selection import train_test_split  # Import train/test splitting.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_gaussian_classification_data(n=180, class_sep=1.3, random_state=123):  # Define a two-class Gaussian-like simulator.
    X, y = make_classification(  # Generate a labeled binary classification data set.
        n_samples=n,  # Set the number of observations.
        n_features=2,  # Keep two predictors so boundaries can be plotted.
        n_informative=2,  # Make both features informative.
        n_redundant=0,  # Avoid redundant features.
        n_clusters_per_class=1,  # Use one cluster per class for clean geometry.
        class_sep=class_sep,  # Control class separation.
        random_state=random_state,  # Fix the simulation seed.
    )  # Finish the simulator call.
    return X, y  # Return predictors and labels.

def make_moon_classification_data(n=200, noise=0.25, random_state=123):  # Define a nonlinear classification simulator.
    X, y = make_moons(  # Generate interlocking moon-shaped classes.
        n_samples=n,  # Set the number of observations.
        noise=noise,  # Add noise to make the problem realistic.
        random_state=random_state,  # Fix the simulation seed.
    )  # Finish the make_moons call.
    return X, y  # Return predictors and labels.

def train_test_accuracy(model, X, y, test_size=0.30, random_state=123):  # Define a local classification train/test evaluator.
    X_train, X_test, y_train, y_test = train_test_split(  # Split before fitting to protect the test set.
        X,  # Pass predictor matrix.
        y,  # Pass class labels.
        test_size=test_size,  # Hold out this fraction for testing.
        random_state=random_state,  # Make the split reproducible.
        stratify=y,  # Preserve class proportions across train and test sets.
    )  # Finish the split call.
    fitted_model = clone(model)  # Clone the model so repeated calls start fresh.
    fitted_model.fit(X_train, y_train)  # Fit using training observations only.
    train_pred = fitted_model.predict(X_train)  # Predict training labels.
    test_pred = fitted_model.predict(X_test)  # Predict test labels.
    train_accuracy = accuracy_score(y_train, train_pred)  # Compute training accuracy.
    test_accuracy = accuracy_score(y_test, test_pred)  # Compute held-out accuracy.
    return fitted_model, train_accuracy, test_accuracy, y_test, test_pred  # Return model, metrics, and test predictions.


In [ ]:
X, y = make_gaussian_classification_data(n=200, class_sep=1.1, random_state=RANDOM_SEED)  # Simulate a binary classification problem.
model = LogisticRegression(max_iter=1000)  # Choose logistic regression as an interpretable baseline classifier.
fitted_model, train_accuracy, test_accuracy, y_test, test_pred = train_test_accuracy(model, X, y, random_state=RANDOM_SEED)  # Fit and evaluate the classifier.
cm = confusion_matrix(y_test, test_pred)  # Compute the confusion matrix on held-out data.
precision = precision_score(y_test, test_pred)  # Compute positive predictive value.
recall = recall_score(y_test, test_pred)  # Compute sensitivity or true-positive rate.
specificity = cm[0, 0] / (cm[0, 0] + cm[0, 1])  # Compute specificity or true-negative rate manually.
summary = pd.DataFrame(  # Build a compact metric table.
    [  # Start the row list.
        {'metric': 'train accuracy', 'value': train_accuracy},  # Add training accuracy.
        {'metric': 'test accuracy', 'value': test_accuracy},  # Add test accuracy.
        {'metric': 'precision', 'value': precision},  # Add precision.
        {'metric': 'recall/sensitivity', 'value': recall},  # Add recall.
        {'metric': 'specificity', 'value': specificity},  # Add specificity.
    ]  # End the row list.
)  # Finish constructing the table.
display(summary)  # Display the metric table.
print('Confusion matrix with rows=true class and columns=predicted class:')  # Explain the matrix orientation.
print(cm)  # Print the confusion matrix.
print(classification_report(y_test, test_pred))  # Print class-specific precision, recall, and F1 summaries.


## Exam-style problem prompt

A prompt reports predicted disease labels. Compute the confusion matrix, accuracy, sensitivity, specificity, and explain the practical cost of false positives versus false negatives.

## Adaptable code pattern

```python
# Step 1: Split data or receive y_true and y_pred.
# Step 2: Compute confusion_matrix(y_true, y_pred).
# Step 3: Compute accuracy, precision, recall, and specificity.
# Step 4: Identify which error type matters for the problem.
# Step 5: Interpret metrics in words, not only numbers.
```

## What to remember for the exam

Know the confusion matrix orientation. Most sklearn outputs use rows for true labels and columns for predicted labels.